# 04 — Outlier Analysis

**Amaç:** Tekil feature outlier'larını, target ilişkisini ve train/test farkını incelemek. Satır silinmez.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from kaggle_s6_e7.config import (FIGURES_DIR, TABLES_DIR, TARGET_COL, ID_COL,
    PLOT_SAMPLE_SIZE, RANDOM_STATE, ensure_report_dirs)
from kaggle_s6_e7.data import load_competition_data, infer_feature_columns, validate_schema
ensure_report_dirs()
train, test = load_competition_data()
validate_schema(train, test)
cat_cols, num_cols = infer_feature_columns(train)
plot_df = train.sample(min(PLOT_SAMPLE_SIZE, len(train)), random_state=RANDOM_STATE)
sns.set_theme(style="whitegrid")

## IQR ve quantile outlier özeti

In [ ]:
from kaggle_s6_e7.eda import outlier_summary
summary=outlier_summary(train,num_cols)
display(summary)
summary.to_csv(TABLES_DIR / "04_outlier_summary.csv")

## Train/test quantile outlier oranları

In [ ]:
from kaggle_s6_e7.features import fit_outlier_bounds, add_outlier_flags
bounds=fit_outlier_bounds(train,num_cols)
train_flags=add_outlier_flags(train,bounds); test_flags=add_outlier_flags(test,bounds)
flag_cols=[c for c in train_flags if c.endswith(("_outlier_low","_outlier_high"))]
comparison=pd.DataFrame({"train_rate":train_flags[flag_cols].mean(),"test_rate":test_flags[flag_cols].mean()})
comparison["delta"]=comparison.test_rate-comparison.train_rate
display(comparison); comparison.to_csv(TABLES_DIR / "04_train_test_outlier_rates.csv")

## Outlier-target ilişkisi

In [ ]:
tables=[]
for col in flag_cols:
    table=pd.crosstab(train_flags[col],train_flags[TARGET_COL],normalize="index")
    table.insert(0,"flag",col); tables.append(table.reset_index())
outlier_target=pd.concat(tables,ignore_index=True)
display(outlier_target)
outlier_target.to_csv(TABLES_DIR / "04_outlier_target.csv",index=False)

## Row-level outlier count

In [ ]:
display(pd.crosstab(train_flags.outlier_count,train_flags[TARGET_COL],normalize="index"))
sns.countplot(data=train_flags,x="outlier_count")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "04_outlier_count.png",dpi=150); plt.show()

## Varsayılan karar
- Outlier nedeniyle satır silme yok.
- Train quantile eşiklerinden flag üretimi V2 adayıdır.
- %0.1–%99.9 clipping yalnız V3 deneyidir.